In [1]:
!pip install h5py
!pip install matplotlib
!pip install transformers
!pip install ipywidgets
!pip install scipy
!pip install kornia torchvision timm


[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# Testing out raw h5 data of Event 11707158 from https://dasway.ess.washington.edu/gci/events/2023-06-10/index.html
import h5py
import matplotlib.pyplot as plt
import os
import numpy as np
import torch
import torch.nn as nn
print(torch.cuda.device_count())
import transformers
from transformers import pipeline
from PIL import Image
import glob

0


In [7]:
from torchvision import transforms


In [ ]:
#modified Anjani's code to test out the BoxSegmenter on the filtered data
from transformers import AutoModelForImageSegmentation
from PIL import Image
import glob
from torchvision import transforms
import torch


device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = AutoModelForImageSegmentation.from_pretrained('briaai/RMBG-2.0', trust_remote_code=True)

torch.set_float32_matmul_precision(['high', 'highest'][0])
model.to(device)
model.eval()

# Data settings
image_size = (8531, 3000)
transform_image = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

image_folder = "C:/Users/alexa/OneDrive/Documents/GitHub/curatedDAS/plots/Output_from_mask/rbg_plots_for_siletzia/normalized-1to1"  # Change this to your folder path
images = glob.glob(os.path.join(image_folder, "*.png"))  

folder = "C:/Users/alexa/OneDrive/Documents/GitHub/curatedDAS/plots/Output_from_mask/rbg_plots_for_siletzia/normalized-1to1/results"

for image_path in images:
    image = Image.open(image_path).convert("RGB")  # Convert image to RGB
    input_images = transform_image(image).unsqueeze(0).to(device)

    # Prediction
    with torch.no_grad():
        preds = model(input_images)[-1].sigmoid().cpu()
    pred = preds[0].squeeze()
    pred_pil = transforms.ToPILImage()(pred)
    mask = pred_pil.resize(image.size)
    image.putalpha(mask)

    # Save the mask as an image and image as well
    image_filename = os.path.join(folder, f"{os.path.basename(image_path).split('.')[0]}_mask_new.png")
    image.save(image_filename)

In [ ]:
'''
model = AutoModelForImageSegmentation.from_pretrained('briaai/RMBG-2.0', trust_remote_code=True)
torch.set_float32_matmul_precision(['high', 'highest'][0])
model.eval()

# Data settings
image_size = (1024, 1024)
transform_image = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

input_images = transform_image(image).unsqueeze(0).to('cuda')

# Prediction
with torch.no_grad():
    preds = model(input_images)[-1].sigmoid().cpu()
pred = preds[0].squeeze()
pred_pil = transforms.ToPILImage()(pred)
mask = pred_pil.resize(image.size)
image.putalpha(mask)

image.save("no_bg_image.png")
'''

In [ ]:
'''
from transformers import AutoModelForAudioClassification,TrainingArguments, Trainer
import datasets

# Load the Whisper model
model = AutoModelForAudioClassification.from_pretrained(args.model_name)

if args.freeze_encoder:
    model.freeze_encoder()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

'''

### MASKRCN_resnet50 model

In [14]:
import torchvision
from torchvision.models.detection import MaskRCNN
from torchvision.models.detection.rpn import AnchorGenerator
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load a pre-trained model for classification and return only the features
backbone = torchvision.models.resnet50(pretrained=True).eval()
backbone = torch.nn.Sequential(*list(backbone.children())[:-2])
backbone.out_channels = 2048


# Generate the anchor generator
rpn_anchor_generator = AnchorGenerator(
    sizes=((32, 64, 128, 256, 512),),
    aspect_ratios=((0.5, 1.0, 2.0),) * 5
)
"""
# Generate the RoI aligner
roi_pooler = torchvision.ops.MultiScaleRoIAlign(
    featmap_names=['0'], output_size=7, sampling_ratio=2
)
"""

# Create the MaskRCNN model
model = MaskRCNN(backbone,
                 num_classes=2,
                 rpn_anchor_generator=rpn_anchor_generator)
                 #box_roi_pool=roi_pooler

model.to(device)
model.eval()


MaskRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn

In [16]:
#setup input images to test the model
image_folder = "C:/Users/alexa/OneDrive/Documents/GitHub/curatedDAS/plots/Output_from_mask/rbg_plots_for_siletzia/normalized0to1"  # Change this to your folder path
images = glob.glob(os.path.join(image_folder, "*.png"))

folder = "C:/Users/alexa/OneDrive/Documents/GitHub/curatedDAS/plots/Output_from_mask/rbg_plots_for_siletzia/normalized0to1/results"

for image_path in images:
    image = Image.open(image_path).convert("RGB")
    input_image = transforms.functional.to_tensor(image).unsqueeze(0).to(device)

    # Perform the prediction
    with torch.no_grad():
        prediction = model(input_image)

    # Extract the masks
    masks = prediction[0]['masks']
    for i, mask in enumerate(masks):
        mask = mask.squeeze().cpu().numpy()
        mask_pil = Image.fromarray((mask * 255).astype(np.uint8))
        mask_pil = mask_pil.resize(image.size)
        image.putalpha(mask_pil)

        # Save the mask as an image
        mask_filename = os.path.join(folder, f"{os.path.basename(image_path).split('.')[0]}_mask_{i}.png")
        image.save(mask_filename)